# Phase 2 fine-tune in Google Colab

Runtime > Change runtime type > GPU.

You need: `phase2_data.zip` (made by `prepare.py`) and `weights/rod_best.pt` (from the repo) uploaded to `Google Drive/MyDrive/phase2/`.

Nothing here retrains from scratch: it starts from `rod_best.pt` and keeps its 25 classes, adding five new ones.

## Cell 1: setup

In [ ]:
!pip -q install ultralytics
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/p2 && cd /content/p2 && unzip -q /content/drive/MyDrive/phase2_data.zip -d data && cp /content/drive/MyDrive/rod_best.pt .
!ls /content/p2 /content/p2/data | head

Mounted at /content/drive
/content/p2:
data
rod_best.pt

/content/p2/data:
phase2_data


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!mv /content/p2/data/phase2_data/* /content/p2/data/ && rmdir /content/p2/data/phase2_data
!ls /content/p2/data

data.yaml  images  labels  labels25  manifest.csv  report.txt


## Cell 2: baseline score of the CURRENT model on the same validation pictures (old 25 classes only)

In [ ]:
from ultralytics import YOLO
import yaml, os
d = '/content/p2/data'
y25 = {'path': d, 'train': 'images/train', 'val': 'images/val', 'names': dict(enumerate(yaml.safe_load(open(d+'/data.yaml'))['names'].values() if False else list(yaml.safe_load(open(d+'/data.yaml'))['names'].values())[:25]))}
# YOLO finds labels by swapping 'images' for 'labels'; for the baseline we need the 25-class label copies, so link them in:
!rm -rf /content/p2/base && mkdir -p /content/p2/base/images /content/p2/base/labels && ln -s /content/p2/data/images/val /content/p2/base/images/val && ln -s /content/p2/data/labels25/val /content/p2/base/labels/val
yaml.safe_dump({'path': '/content/p2/base', 'train': 'images/val', 'val': 'images/val', 'names': y25['names']}, open('/content/p2/base25.yaml', 'w'))
base = YOLO('/content/p2/rod_best.pt').val(data='/content/p2/base25.yaml', imgsz=416, plots=False)
print('BASELINE mAP50 per class:', dict(zip(base.names.values(), base.box.ap50.round(3))))

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics 8.4.193 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,138,539 parameters, 0 gradients, 8.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2122.0±633.1 MB/s, size: 58.9 KB)
val: Scanning /content/p2/data/labels/val... 965 images, 0 backgrounds, 282 corrupt: 100% ━━━━━━━━━━━━ 965/965 2.0Kit/s 0.5s
val: /content/p2/data/images/val/blind_assist_000000.jpg: ignoring corrupt image/label: Label class 25 exceeds dataset class count 25. Possible class labels are 0-24
val: /content/p2/data/images/val/blind_assist_000009.jpg: ignoring corrupt image/label: Label class 25 exceeds dataset class count 25. Possible class labels are 0-24
va

## Cell 3: keep what the old model knows (pseudo-label the old 25 classes)

In [ ]:
# Most of the new pictures have no boxes for trees, manholes, poles... The old model fills those in, so training does not erase them.
import glob
old = YOLO('/content/p2/rod_best.pt')
def iou(a, b):
    ax1, ay1, ax2, ay2 = a[0]-a[2]/2, a[1]-a[3]/2, a[0]+a[2]/2, a[1]+a[3]/2
    bx1, by1, bx2, by2 = b[0]-b[2]/2, b[1]-b[3]/2, b[0]+b[2]/2, b[1]+b[3]/2
    iw, ih = max(0, min(ax2, bx2)-max(ax1, bx1)), max(0, min(ay2, by2)-max(ay1, by1)); i = iw*ih
    return i / (a[2]*a[3] + b[2]*b[3] - i + 1e-9)
added = 0
for split in ('train',):
    for img in glob.glob(f'{d}/images/{split}/*'):
        lbl = img.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'
        have = [tuple(map(float, l.split())) for l in open(lbl)] if os.path.exists(lbl) else []
        r = old.predict(img, imgsz=416, conf=0.55, verbose=False)[0]
        new = []
        for c, xywhn, cf in zip(r.boxes.cls.tolist(), r.boxes.xywhn.tolist(), r.boxes.conf.tolist()):
            if any(iou(xywhn, h[1:]) > 0.4 for h in have): continue   # something is already labelled there (maybe by a human)
            new.append(f'{int(c)} ' + ' '.join(f'{v:.6f}' for v in xywhn) + '\n'); added += 1
        if new: open(lbl, 'a').writelines(new)
print('pseudo-labels added:', added)

pseudo-labels added: 1071


## Cell 4: stage 1, new head only (backbone frozen) then stage 2, everything gently

In [ ]:
!sed -i 's#^path: .*#path: /content/p2/data#' /content/p2/data/data.yaml
!head -3 /content/p2/data/data.yaml

path: /content/p2/data
train: images/train
val: images/val


In [ ]:
m = YOLO('/content/p2/rod_best.pt')
m.train(data=d+'/data.yaml', imgsz=416, epochs=15, batch=32, freeze=10, lr0=0.002, project='/content/p2/runs', name='s1', patience=0)
m = YOLO('/content/p2/runs/s1/weights/best.pt')
m.train(data=d+'/data.yaml', imgsz=416, epochs=40, batch=32, lr0=0.0005, project='/content/p2/runs', name='s2', patience=10)

Ultralytics 8.4.193 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/p2/data/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=15, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, iou=0.7, kobj=1.0, line_width=None, lr0=0.002, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/p2/rod_best.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=s1, nbs=64, nms=None, opset=None, optimize=False, o

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0,  2,  3,  4,  8,  9, 10, 14, 15, 16, 17, 19, 21, 23, 25, 26, 27, 28, 29])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7b209fb62580>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,

In [16]:
from google.colab import drive, files
drive.mount('/content/drive')
import shutil, os
from ultralytics import YOLO
os.makedirs('/content/drive/MyDrive/phase2', exist_ok=True)
for f in ('best.pt', 'last.pt'):
    shutil.copy(f'/content/p2/runs/s2/weights/{f}', f'/content/drive/MyDrive/phase2/rod2_{f}')
YOLO('/content/p2/runs/s2/weights/best.pt').export(format='onnx', imgsz=416, opset=12, simplify=True)
shutil.copy('/content/p2/runs/s2/weights/best.onnx', '/content/drive/MyDrive/phase2/rod2_416.onnx')
files.download('/content/p2/runs/s2/weights/best.onnx')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Ultralytics 8.4.193 🚀 Python-3.13.16 torch-2.11.0+cu130 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino
Model summary (fused): 73 layers, 3,139,754 parameters, 0 gradients, 3.7 GFLOPs

PyTorch: starting from '/content/p2/runs/s2/weights/best.pt' with input shape (1, 3, 416, 416) BCHW and output shape(s) (1, 34, 3549) (6.2 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxruntime', 'onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.13.16 environment at: /usr
Resolved 12 packages in 187ms
Prepared 4 packages in 541ms
Installed 4 packages in 43ms
 + colorama==0.4.6
 + onnx==1.23.2
 + onnxruntime==1.31.1
 + onnxslim==0.1.98

requirements: AutoUpdate success ✅ 1.1s
WARNING ⚠️ requirements: Restart run

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Cell 5: compare to the baseline, class by class

In [18]:
from ultralytics import YOLO
import yaml, os, shutil
d = '/content/p2/data'

names = yaml.safe_load(open(d+'/data.yaml'))['names']
names = list(names.values()) if isinstance(names, dict) else list(names)

# real copies, not symlinks, so YOLO reads the 25-class labels
shutil.rmtree('/content/p2/base', ignore_errors=True)
os.makedirs('/content/p2/base/images'); os.makedirs('/content/p2/base/labels')
shutil.copytree(d+'/images/val', '/content/p2/base/images/val')
shutil.copytree(d+'/labels25/val', '/content/p2/base/labels/val')
yaml.safe_dump({'path': '/content/p2/base', 'train': 'images/val', 'val': 'images/val',
                'names': dict(enumerate(names[:25]))}, open('/content/p2/base25.yaml', 'w'))

def per_class(r):  # scores only exist for classes present in val, so match by class index
    return {r.names[int(i)]: float(a) for i, a in zip(r.box.ap_class_index, r.box.ap50)}

base = YOLO('/content/p2/rod_best.pt').val(data='/content/p2/base25.yaml', imgsz=416, plots=False, verbose=False)
new  = YOLO('/content/p2/runs/s2/weights/best.pt').val(data=d+'/data.yaml', imgsz=416, plots=False, verbose=False)
b, n = per_class(base), per_class(new)

print(f'{"class":22s} {"old":>7s} {"new":>7s} {"change":>8s}')
for k in n:
    o = b.get(k)
    ch = '' if o is None else f'{n[k]-o:+.3f}'
    flag = '  <-- dropped' if o is not None and n[k]-o < -0.03 else ''
    print(f'{k:22s} {("-" if o is None else f"{o:.3f}"):>7s} {n[k]:7.3f} {ch:>8s}{flag}')

Ultralytics 8.4.193 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,138,539 parameters, 0 gradients, 8.7 GFLOPs
val: Fast image access ✅ (ping: 0.2±0.4 ms, read: 1046.2±704.4 MB/s, size: 82.4 KB)
val: Scanning /content/p2/base/labels/val... 965 images, 282 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 965/965 1.1Kit/s 0.9s
val: New cache created: /content/p2/base/labels/val.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 61/61 10.7it/s 5.7s
                   all        965       2062       0.21      0.141      0.131     0.0877
Speed: 0.5ms preprocess, 2.0ms inference, 0.0ms loss, 1.4ms postprocess per image
Ultralytics 8.4.193 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,139,754 parameters, 0 gradients, 8.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2090.8±1169.4 MB/s, size: 57.7 KB)
val: Scanning /conten

## Cell 6: export for the app, and download

In [19]:
from google.colab import drive, files
drive.mount('/content/drive')
import os, shutil
from ultralytics import YOLO

W = '/content/p2/runs/s2/weights'
out = '/content/drive/MyDrive/phase2'
os.makedirs(out, exist_ok=True)

# keep the trained weights safe on Drive
for f in ('best.pt', 'last.pt'):
    shutil.copy(f'{W}/{f}', f'{out}/rod2_{f}')

# export for the app (416 px, same settings as the app's other models)
YOLO(f'{W}/best.pt').export(format='onnx', imgsz=416, opset=12, simplify=True)
shutil.copy(f'{W}/best.onnx', f'{out}/rod2_416.onnx')

# quick check: output should be [1, 34, 3549] (4 box values + 30 classes)
import onnx
g = onnx.load(f'{out}/rod2_416.onnx').graph
print('output shape:', [x.dim_value for x in g.output[0].type.tensor_type.shape.dim])
print('size MB:', round(os.path.getsize(f'{out}/rod2_416.onnx') / 1e6, 1))

files.download(f'{out}/rod2_416.onnx')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Ultralytics 8.4.193 🚀 Python-3.13.16 torch-2.11.0+cu130 CPU (Intel Xeon CPU @ 2.00GHz)
Model summary (fused): 73 layers, 3,139,754 parameters, 0 gradients, 3.7 GFLOPs

PyTorch: starting from '/content/p2/runs/s2/weights/best.pt' with input shape (1, 3, 416, 416) BCHW and output shape(s) (1, 34, 3549) (6.2 MB)

ONNX: starting export with onnx 1.23.2 opset 12...
ONNX: slimming with onnxslim 0.1.98...
ONNX: export success ✅ 1.1s, saved as '/content/p2/runs/s2/weights/best.onnx' (12.1 MB)

Export complete (1.3s)
Results saved to /content/p2/runs/s2/weights/best.onnx
Predict:         yolo predict task=detect model=/content/p2/runs/s2/weights/best.onnx imgsz=416 
Validate:        yolo val task=detect model=/content/p2/runs/s2/weights/best.onnx imgsz=416 data=/content/p2/data/data.yaml 
Visualize:       https://netron.app
output shape: [1, 34, 3549]
size MB: 12.7


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>